In [14]:
import glob
import json
import os
import re
import unicodedata
from typing import List

from bs4 import BeautifulSoup
import detectron2
import easyocr
import fitz
from grobid_client.grobid_client import GrobidClient
import layoutparser as lp
import numpy as np
import ollama
import pandas as pd
from pdf2image import convert_from_path
from pydantic import BaseModel
import requests
from tqdm import tqdm


In [15]:
pdf_location = os.path.join(os.getcwd(), "starting_point")
pdf_files = glob.glob(os.path.join(pdf_location, "*.pdf"))
print(f"Found {len(pdf_files)} PDF files.")

Found 1 PDF files.


In [ ]:


def extract_bibliography(pdf_path: str, grobid_url: str = "http://localhost:8070"):
    endpoint = f"{grobid_url}/api/processReferences"

    with open(pdf_path, "rb") as f:
        # consolidateCitations: "1" queries CrossRef to fill missing DOIs/dates; "0" is faster & offline
        response = requests.post(
            endpoint, files={"input": f}, data={"consolidateCitations": "0"}
        )

    if response.status_code != 200:
        raise RuntimeError(f"GROBID error {response.status_code}: {response.text}")

    soup = BeautifulSoup(response.text, "xml")
    references = []

    for bib in soup.find_all("biblStruct"):
        # 1. Article / Book Title
        title_tag = bib.find("title", level="a") or bib.find("title", level="m")
        title = title_tag.text.strip() if title_tag else None

        # 2. Journal / Conference / Monograph Title
        journal_tag = bib.find("title", level="j") or bib.find(
            "title", level="m" if bib.find("title", level="a") else None
        )
        journal = (
            journal_tag.text.strip()
            if journal_tag and journal_tag != title_tag
            else None
        )

        # 3. Authors
        authors = []
        for author in bib.find_all("author"):
            pers_name = author.find("persName")
            if pers_name:
                first = (
                    pers_name.find("forename", type="first").text.strip()
                    if pers_name.find("forename", type="first")
                    else ""
                )
                middle = (
                    pers_name.find("forename", type="middle").text.strip()
                    if pers_name.find("forename", type="middle")
                    else ""
                )
                last = (
                    pers_name.find("surname").text.strip()
                    if pers_name.find("surname")
                    else ""
                )
                full_name = " ".join(
                    part for part in [first, middle, last] if part
                )
                if full_name:
                    authors.append(full_name)

        # 4. Publication Year
        date_tag = bib.find("date", type="published") or bib.find("date")
        year = None
        if date_tag:
            year = date_tag.get("when") or date_tag.text.strip()
            year = year[:4] if year else None

        # 5. Volume, Issue, Pages
        volume = bib.find("biblScope", unit="volume")
        issue = bib.find("biblScope", unit="issue")
        page = bib.find("biblScope", unit="page")

        # 6. DOI / Identifiers
        doi_tag = bib.find("idno", type="DOI") or bib.find("idno", type="doi")
        doi = doi_tag.text.strip() if doi_tag else None

        # Raw citation string (if captured)
        raw_note = bib.find("note", type="raw_reference")
        raw_text = raw_note.text.strip() if raw_note else None

        references.append(
            {
                "title": title,
                "authors": authors,
                "journal_or_venue": journal,
                "year": year,
                "volume": volume.text.strip() if volume else None,
                "issue": issue.text.strip() if issue else None,
                "pages": page.text.strip() if page else None,
                "doi": doi,
                "raw_reference": raw_text,
            }
        )

    return references

In [13]:
refs = extract_bibliography(pdf_location + "/p1.pdf")

# Export to JSON
with open("references.json", "w", encoding="utf-8") as out:
    json.dump(refs, out, indent=2, ensure_ascii=False)

print(f"Successfully extracted {len(refs)} references.")

Successfully extracted 56 references.
